# Data Preprocessing

[click here for dataset](https://drive.google.com/file/d/1_kxlGz-Ml4tcDdGFyZLf8PwpKvpr1q-v/view?usp=drivesdk)

In [2]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Perceptron, SGDClassifier, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('/content/Copy of V1 (pre) (1).csv')
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,IncomeGroup
0,62,Self-emp-not-inc,26911,7th-8th,4,Widowed,Other-service,Not-in-family,White,Female,0,0,66,United-States,<=50K
1,18,Private,208103,11th,7,Never-married,Other-service,Other-relative,White,Male,0,0,25,United-States,<=50K
2,25,Private,102476,Bachelors,13,Never-married,Farming-fishing,Own-child,White,Male,27828,0,50,United-States,>50K
3,33,Private,511517,HS-grad,9,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,40,United-States,<=50K
4,36,Private,292570,11th,7,Never-married,Machine-op-inspct,Unmarried,White,Female,0,0,40,United-States,<=50K


## Q1. Which dataset are you using for this exam?
- V1
- V2
- V3
- V4
- V5

In [3]:
# Answer: V1
print("Dataset used: V1")

Dataset used: V1


## Q2. How many total number of rows and columns are there in the dataset?
- Rows:2000 , Columns: 15
- Rows:20000 , Columns: 14
- Rows:20000 , Columns: 16
- Rows:20000 , Columns: 15


\

Answer: D

In [4]:
print("Shape of dataset:", df.shape)
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

Shape of dataset: (20000, 15)
Rows: 20000, Columns: 15


## Q3. Is there any Missing values in the given dataset ?
- Yes
- No


\

Answer: Yes

In [5]:
# Check for missing values (including '?' symbols)
print("Any NaN values?", df.isnull().any().any())
print("\nChecking for '?' values in each column:")
for col in df.columns:
    count = (df[col].astype(str).str.strip() == '?').sum()
    if count > 0:
        print(f"  {col}: {count} '?' found")
print("\nAnswer: Yes — '?' symbols represent missing values")

Any NaN values? False

Checking for '?' values in each column:
  workclass: 1137 '?' found
  occupation: 1140 '?' found
  native-country: 354 '?' found

Answer: Yes — '?' symbols represent missing values


## Q4. Check all " ?" symbol in the dataset and replace it with "np.nan" value . Which of the columns in the dataset have null values?
- age
- native-country
- capital-gain
- workclass
- occupation
- None of these


\


Answer: B, D, E

In [6]:
# Replace ' ?' with np.nan
df.replace(' ?', np.nan, inplace=True)

print("Missing values per column after replacement:")
print(df.isnull().sum())
print("\nColumns with null values: native-country, workclass, occupation")

Missing values per column after replacement:
age                  0
workclass         1137
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        1140
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     354
IncomeGroup          0
dtype: int64

Columns with null values: native-country, workclass, occupation


## Q5. What fraction of total samples (in percentage) have missing value in occupation column?

\
Answer: 5.7%

In [7]:
pct_missing_occupation = df['occupation'].isnull().mean() * 100
print(f"Fraction of samples with missing occupation: {pct_missing_occupation:.1f}%")

Fraction of samples with missing occupation: 5.7%


## Q6. What fraction of total samples(In percentage) have income <= 50k in the given dataset? (round the value upto 2 decimal places)
- 75.95
- 75.83
- 76.51
- 72.82
- 76.00


\

Answer: C

In [8]:
pct_leq_50k = (df['IncomeGroup'].str.strip() == '<=50K').mean() * 100
print(f"Fraction with income <=50K: {pct_leq_50k:.2f}%")

Fraction with income <=50K: 76.51%


## Q7: What is the mean age of the samples given in the dataset(mark the closest integer from the given options)?
- 25
- 34
- 38
- 45
- 50


\

Answer: C

In [9]:
mean_age = df['age'].mean()
print(f"Mean age: {mean_age:.2f}")
print("Closest option: 38")

Mean age: 38.55
Closest option: 38


## Q8: How many people have completed their education upto "Preschool" only ?
- 29
- 32
- 26
- 27


\

Answer: B

In [10]:
preschool_count = (df['education'].str.strip() == 'Preschool').sum()
print(f"Number of people with Preschool education: {preschool_count}")

Number of people with Preschool education: 32


## Q9. What is the correlation cofficient between column "education-num" and column "capital-gain" ?


\

Answer: 0.12

In [11]:
corr = df['education-num'].corr(df['capital-gain'])
print(f"Correlation between education-num and capital-gain: {corr:.2f}")

Correlation between education-num and capital-gain: 0.12


## Apply SimpleImputer with strategy = 'most_frequent' to replace all NaN value by mode of respective column in the original dataset.(Use this updated dataset only for all further questions)

## Q10. What is the mode of 'occupation' column of updated datset?
- Adm-clerical
- Prof-specialty
- Exec-managerial
- Craft-repair
- Private


\

Answer: B

In [12]:
# Apply SimpleImputer with most_frequent strategy
imputer = SimpleImputer(strategy='most_frequent')
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

print("Mode of 'occupation' column:", df_imputed['occupation'].mode()[0])

Mode of 'occupation' column:  Prof-specialty


## Split the dataset into X1, X2 and y, where X1 contains all Numerical features, X2 contains all categorical features(except "IncomeGroup") and save 'IncomeGroup' column in y variable. Then apply OneHotEncoder on the categorical features(X2) with option (sparse = False) and StandardScaler on the numerical features (X2).


## Q11. What is the total number of columns in X1 and X2 variables respectively?
- (6,9)
- (9,6)
- (6,8)
- (8,6)


\

Answer: C

In [13]:
y = df_imputed['IncomeGroup']

# Numerical and categorical features
numerical_cols = df_imputed.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in df_imputed.columns if c not in numerical_cols and c != 'IncomeGroup']

X1 = df_imputed[numerical_cols]
X2 = df_imputed[categorical_cols]

print(f"X1 (numerical) columns: {len(numerical_cols)} -> {numerical_cols}")
print(f"X2 (categorical) columns: {len(categorical_cols)} -> {categorical_cols}")
print(f"\nAnswer: X1={len(numerical_cols)}, X2={len(categorical_cols)}")

X1 (numerical) columns: 0 -> []
X2 (categorical) columns: 14 -> ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']

Answer: X1=0, X2=14


## Apply StandardScaler on the X1 data and OneHotEncoder on X2 data with option (sparse = False).

## Q12. What are the data types of X1 and X2?
- dataframe, dataframe
- dataframe, numpy array
- dense scipy array, numpy array
- numpy array, numpy array


\

Answer: D

In [14]:
# Re-define numerical and categorical features and ensure correct dtypes,
# as the previous cell might have incorrectly identified them due to type issues
# after SimpleImputer output.

# Define the numerical and categorical features based on the dataset structure and Q11
numerical_features = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_features = [col for col in df_imputed.columns if col not in numerical_features and col != 'IncomeGroup']

# Ensure numerical columns in df_imputed are of numeric type
# This step is crucial if SimpleImputer converted them to 'object' dtype
for col in numerical_features:
    df_imputed[col] = pd.to_numeric(df_imputed[col], errors='coerce') # Use coerce to handle any non-numeric gracefully

# Re-assign X1 and X2 based on correct feature types
X1 = df_imputed[numerical_features]
X2 = df_imputed[categorical_features]

scaler = StandardScaler()
X1 = scaler.fit_transform(X1)

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore') # Changed sparse to sparse_output for newer sklearn
X2 = ohe.fit_transform(X2)

print(f"Type of X1: {type(X1)}")
print(f"Type of X2: {type(X2)}")
print("Both are numpy arrays -> Answer: D")

Type of X1: <class 'numpy.ndarray'>
Type of X2: <class 'numpy.ndarray'>
Both are numpy arrays -> Answer: D


## Concatenate X1 and X2 and call it X( Keep axis = 1) and then Convert X to a dataframe.
## Q13. What is the new shape of resultant X?

- (20000, 105)
- (20000, 204)
- (20000, 14)
- (2000, 98)


\

Answer: A

In [15]:
X = np.concatenate([X1, X2], axis=1)
X = pd.DataFrame(X)
print(f"Shape of X: {X.shape}")

Shape of X: (20000, 105)


# Model Building

Make use of your preprocessed data that you got from above preprocessing steps.

Note: In current OPPE setup you will be given the preprocessed dataset for Model Building.

## Q14. Split the dataset into training and validation dataset into 80:20 ratio while keeping random_state =64. what is the shape of the y_train dataset?
- (20000,)
- (16000, 15)
- (20000, 14)
- (4000, 98)
- (16000,)



\

Answer: D

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=64)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test:  {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test:  {y_test.shape}")
print("Answer: E -> (16000,)")

Shape of X_train: (16000, 105)
Shape of X_test:  (4000, 105)
Shape of y_train: (16000,)
Shape of y_test:  (4000,)
Answer: E -> (16000,)


## Instantiate a perceptron classifier that with following parameters:
- Fit the intercept
- Put warm start to be False

## Fit this perceptron model with the training dataset and calculate the accuracy score for training data.
## Q15: Which of the following option matches with the score?
- 0.8130625
- 0.8379375
- 0.8239375
- 0.8058125
- 0.78475



\

Answer: A

In [17]:
perceptron1 = Perceptron(fit_intercept=True, warm_start=False)
perceptron1.fit(X_train, y_train)

train_acc = accuracy_score(y_train, perceptron1.predict(X_train))
print(f"Training accuracy: {train_acc}")

Training accuracy: 0.813


## Q16. Create a new Perceptron(random_state=42) object with given settings in the instrunctions and train it on training set . What is the value of bias (intercept) (upto 1 decimal point) ?
- Set early stopping and fit intercept to be True
- Put warm start to be False



\

Answer: -8

In [18]:
perceptron2 = Perceptron(random_state=42, early_stopping=True, fit_intercept=True, warm_start=False)
perceptron2.fit(X_train, y_train)

print(f"Intercept (bias): {perceptron2.intercept_}")
print(f"Rounded to 1 decimal: {round(perceptron2.intercept_[0], 1)}")

Intercept (bias): [-8.]
Rounded to 1 decimal: -8.0


## Use SGDClassifier on the training dataset (X_train and y_train) to train the model. Use the following parameters:

- log is the loss function to be used
- apply ridge regularization,
- maximum number of passes over the training data is 10
- initial learning rate is 0.01,
- regularization rate value is 0.001,
- learning rate should not change during training.
- Take random_state=64.
- Set warm_state as False

Note : Please ignore the convergence warning.

## Q17. Based on this operation, calculate and enter the correct value of accuracy (Upto 4 decimal points).

ANS: (Type: Range) 0.840,0.855

In [19]:
sgd = SGDClassifier(
    loss='log_loss',
    penalty='l2',
    max_iter=10,
    eta0=0.01,
    alpha=0.001,
    learning_rate='constant',
    random_state=64,
    warm_start=False
)
sgd.fit(X_train, y_train)

sgd_acc = accuracy_score(y_test, sgd.predict(X_test))
print(f"SGDClassifier accuracy on test set: {sgd_acc:.4f}")

SGDClassifier accuracy on test set: 0.8482


## Take LogisticRegression estimator with following parameters for fitting on the training dataset:
- Use sag as solver
- Set random state to be equal to 64
- Tolerance for stopping criteria to be 1e-3
- Maximum number of iterations taken for the solvers to converge to be 100

##Q18. Enter the recall score you got for the given model (Take average as macro).

ANS: (Type: Range) 0.74,0.78

In [20]:
lr = LogisticRegression(solver='sag', random_state=64, tol=1e-3, max_iter=100)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
recall = recall_score(y_test, y_pred_lr, average='macro')
print(f"Recall score (macro): {recall:.4f}")

Recall score (macro): 0.7529


## Use Gridsearchcv with KNeighborsClassifier() being the estimator, accuracy as scoring parameter, cv value as 4 and consider [1,3,5,7] as "number of neighbors" to be examined.

## Consider following parameters for KNeighborsClassifier():
- Take metric as 'minkowski',
- Set P value as 2
- Keep other parameter values as default value.

## Q19. What is the best value of K you obtained using above instruction?



\

Answer: 7

In [21]:
knn = KNeighborsClassifier(metric='minkowski', p=2)
param_grid_knn = {'n_neighbors': [1, 3, 5, 7]}

grid_knn = GridSearchCV(knn, param_grid_knn, scoring='accuracy', cv=4)
grid_knn.fit(X_train, y_train)

print(f"Best K (n_neighbors): {grid_knn.best_params_['n_neighbors']}")
print(f"Best score: {grid_knn.best_score_:.4f}")

Best K (n_neighbors): 7
Best score: 0.8348


#(Common Instruction for Q20 to Q23)
## Take DecisionTreeClassifier(random_state = 64) with GridSearchCV to train the model.Hyperparameter tuning to be done over the following parameters:
- Use criterion as 'entropy' or 'gini'
- Use splitter as 'random' or 'best'
- Use minimum number of samples per leaf as [2,4,6,8,10]
- Use maximum depth as [3,4,5,6]
- Use cross validation = 4

## Q20. Enter the value (up to 2 decimal points) of the 'score' on testing set using best model.



\

Answer: 0.84

In [22]:
dt = DecisionTreeClassifier(random_state=64)
param_grid_dt = {
    'criterion': ['entropy', 'gini'],
    'splitter': ['random', 'best'],
    'min_samples_leaf': [2, 4, 6, 8, 10],
    'max_depth': [3, 4, 5, 6]
}

grid_dt = GridSearchCV(dt, param_grid_dt, cv=4)
grid_dt.fit(X_train, y_train)

best_dt = grid_dt.best_estimator_
test_score = best_dt.score(X_test, y_test)
print(f"Best params: {grid_dt.best_params_}")
print(f"Test score: {test_score:.2f}")

Best params: {'criterion': 'gini', 'max_depth': 6, 'min_samples_leaf': 10, 'splitter': 'best'}
Test score: 0.85


## Q21. Enter the value of best max_depth of the model after training with GridSearchCV.



\

Answer: 6

In [23]:
print(f"Best max_depth: {grid_dt.best_params_['max_depth']}")

Best max_depth: 6


## Q22. Enter the value of best min_samples_leaf of the model after training with GridSearchCV.



\

Answer: 10

In [24]:
print(f"Best min_samples_leaf: {grid_dt.best_params_['min_samples_leaf']}")

Best min_samples_leaf: 10


## Q23: What are the number of nodes in the optimal tree?



\

Answer: 75

In [25]:
print(f"Number of nodes in the best decision tree: {best_dt.tree_.node_count}")

Number of nodes in the best decision tree: 75


## Take RandomForestClassifier (random state to be 64) with GridSearchCV to tune the number of decision trees with training set. The number of trees in forest can range from 5 to 10 (both inclusive).
## Q24. Mark the number of decision trees that will produce the best score on the training data.
- 5
- 7
- 8
- 10
- 9
- 6



\

Answer: D

In [26]:
rf = RandomForestClassifier(random_state=64)
param_grid_rf = {'n_estimators': list(range(5, 11))}

grid_rf = GridSearchCV(rf, param_grid_rf, cv=4)
grid_rf.fit(X_train, y_train)

print(f"Best n_estimators: {grid_rf.best_params_['n_estimators']}")
print(f"Best cross-val score: {grid_rf.best_score_:.4f}")

Best n_estimators: 10
Best cross-val score: 0.8453


# (Common Instructions for Q25,Q26)

## Take an adaboost model with following hyperparameter values and tune it using GridsearchCV.
- Use n_estimators as [10,20,30]
- random_state = 64
- Use learning_rate as [0.5,1,2]
- Take cv value= 4

## Q25. Train the 'model' using above instructions and enter the 'accuracy score' (up to 3 decimal points) on the test data.



\

Answer: 0.85

In [27]:
ada = AdaBoostClassifier(random_state=64)
param_grid_ada = {
    'n_estimators': [10, 20, 30],
    'learning_rate': [0.5, 1, 2]
}

grid_ada = GridSearchCV(ada, param_grid_ada, cv=4)
grid_ada.fit(X_train, y_train)

best_ada = grid_ada.best_estimator_
ada_acc = accuracy_score(y_test, best_ada.predict(X_test))
print(f"Best params: {grid_ada.best_params_}")
print(f"Accuracy on test data: {ada_acc:.3f}")

Best params: {'learning_rate': 1, 'n_estimators': 30}
Accuracy on test data: 0.852


## Q26. Enter the value of best n_estimators of the model after training with GridSearchCV.



\

Answer: 30

In [28]:
print(f"Best n_estimators: {grid_ada.best_params_['n_estimators']}")

Best n_estimators: 30


## Apply GridsearchCV and support vector machine (SVM)(kernel':('linear', 'rbf'), 'C':[1, 10]) on the training dataset X_train, y_train and calculate the best value of C and kernel.

## Q27. Which of the following options represent the best parameters. (Keep patience: It is common if it takes arround 5 min to complete the run.)

- {'C': 1, 'kernel': 'rbf'}
- {'C': 10, 'kernel': 'rbf'}
- {'C': 10, 'kernel': 'linear'}
- {'C': 1, 'kernel': 'linear'}
- None of these

\

Answer: A


In [29]:
svm = SVC()
param_grid_svm = {
    'kernel': ['linear', 'rbf'],
    'C': [1, 10]
}

grid_svm = GridSearchCV(svm, param_grid_svm, cv=4)
grid_svm.fit(X_train, y_train)

print(f"Best SVM params: {grid_svm.best_params_}")
print(f"Best cross-val score: {grid_svm.best_score_:.4f}")

Best SVM params: {'C': 1, 'kernel': 'rbf'}
Best cross-val score: 0.8562
